# 02 Strategy Backtest Portfolio

This notebook demonstrates **ETL** usage for a real market pull and the signal -> intent -> risk -> execution -> portfolio chain with ledger visibility.


In [ ]:
from algotradeplan.data import ETL
from src.algotradeplan.orchestration.trade_flow import TradeFlow
from src.algotradeplan.plugins.connectors.simulated_fill_connector import SimulatedFillExecutionConnectorPlugin
from src.algotradeplan.plugins.risk.engine import RiskEngine
from src.algotradeplan.plugins.strategies.ema_cross_atr_stop import EmaCrossAtrStopStrategyPlugin
from src.algotradeplan.portfolio.manager import PortfolioManager

etl = ETL()
df = etl.load_market_data(
    source="binance_futures",
    symbol="BTCUSDT",
    dataset="kline",
    timeframe="1m",
    limit=200,
)

strategy = EmaCrossAtrStopStrategyPlugin()
portfolio = PortfolioManager(starting_cash=10_000.0)
flow = TradeFlow(
    strategy=strategy,
    risk=RiskEngine(max_notional=2_000.0),
    execution=SimulatedFillExecutionConnectorPlugin(),
)

# ledger
flow_result = flow.run({"symbol": "BTCUSDT", "price": 100.0, "quantity": 0.01, "candles": [{"high": 101, "low": 99, "close": 100}] * 25})
portfolio_snapshot = portfolio.apply_execution(flow_result.execution)
ledger = portfolio.ledger()

{"rows": len(df) if hasattr(df, "__len__") else 0, "execution": flow_result.execution, "portfolio": portfolio_snapshot, "ledger": ledger}
